In [1]:
# ============================================================
# FINAL FAIR XLM-R TEST EVALUATION
#
# CONDITIONS:
# 1. Original RUHSOLD XLM-R
# 2. Corrected Unfiltered 1x XLM-R
# 3. Corrected QC-Filtered 1x XLM-R
#
# IMPORTANT:
# - Same untouched RUHSOLD test set for every model
# - Same tokenizer
# - Same max length
# - Same inference implementation
# - Same metric implementation
# - Same class order
# - All checkpoints loaded normally for evaluation
# - Test set is used ONLY here
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


FINAL_TEST_RESULTS_ROOT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "final_corrected_xlmr_comparison"
)


FINAL_TEST_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MODEL / DATA CONFIGURATION
# ============================================================

MODEL_NAME = (
    "FacebookAI/xlm-roberta-base"
)

MAX_LENGTH = 128

NUM_LABELS = 5

TEST_BATCH_SIZE = 4


METRIC_LABELS = [
    0,
    1,
    2,
    3,
    4,
]


id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}


label2id = {
    label: class_id
    for class_id, label
    in id2label.items()
}


# ============================================================
# FINAL CHECKPOINT REGISTRY
#
# THESE ARE THE VALIDATION-SELECTED BEST CHECKPOINTS.
# ============================================================

FINAL_CHECKPOINTS = {

    # --------------------------------------------------------
    # ORIGINAL RUHSOLD XLM-R
    # --------------------------------------------------------

    "Original XLM-R": {

        42: (
            PROJECT_ROOT
            / "outputs"
            / "xlm_roberta_original_final_3seed"
            / "diagnostic_seed_42"
            / "checkpoint-2005"
        ),

        43: (
            PROJECT_ROOT
            / "outputs"
            / "xlm_roberta_original_final_3seed"
            / "seed_43"
            / "checkpoint-2005"
        ),

        44: (
            PROJECT_ROOT
            / "outputs"
            / "xlm_roberta_original_final_3seed"
            / "seed_44"
            / "checkpoint-1203"
        ),
    },


    # --------------------------------------------------------
    # CORRECTED UNFILTERED 1x
    # --------------------------------------------------------

    "Unfiltered 1x": {

        42: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_unfiltered_1x_corrected"
            / "seed_42"
            / "checkpoint-4419"
        ),

        43: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_unfiltered_1x_corrected"
            / "seed_43"
            / "checkpoint-1964"
        ),

        44: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_unfiltered_1x_corrected"
            / "seed_44"
            / "checkpoint-1473"
        ),
    },


    # --------------------------------------------------------
    # CORRECTED QC-FILTERED 1x
    # --------------------------------------------------------

    "QC-Filtered 1x": {

        42: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_filtered_1x_corrected"
            / "seed_42"
            / "checkpoint-982"
        ),

        43: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_filtered_1x_corrected"
            / "seed_43"
            / "checkpoint-4419"
        ),

        44: (
            PROJECT_ROOT
            / "classifier"
            / "outputs"
            / "xlm_roberta_filtered_1x_corrected"
            / "seed_44"
            / "checkpoint-3437"
        ),
    },
}


# ============================================================
# VERIFY ALL NINE CHECKPOINTS BEFORE TOUCHING TEST SET
# ============================================================

print("=" * 80)
print("FINAL XLM-R CHECKPOINT AUDIT")
print("=" * 80)


for condition, seed_map in FINAL_CHECKPOINTS.items():

    print(
        f"\n{condition}"
    )

    for seed, checkpoint in seed_map.items():

        exists = checkpoint.exists()

        print(
            f"Seed {seed}:",
            exists,
            checkpoint
        )

        assert exists, (
            f"Missing checkpoint:\n{checkpoint}"
        )


print(
    "\nAll nine final XLM-R checkpoints verified."
)


# ============================================================
# LOAD UNTOUCHED TEST SET
#
# RUHSOLD files are headerless.
# ============================================================

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    header=None,
    names=[
        "tweet",
        "label",
    ],
)


test_df["tweet"] = (
    test_df["tweet"]
    .astype(str)
)


test_df["label"] = (
    test_df["label"]
    .astype(int)
)


# ============================================================
# TEST-SET INTEGRITY CHECK
# ============================================================

EXPECTED_TEST_SIZE = 2003


EXPECTED_TEST_COUNTS = {
    0: 481,
    1: 1070,
    2: 156,
    3: 168,
    4: 128,
}


assert (
    len(test_df)
    ==
    EXPECTED_TEST_SIZE
)


assert (
    test_df[
        "tweet"
    ]
    .isna()
    .sum()
    ==
    0
)


assert (
    test_df[
        "label"
    ]
    .isna()
    .sum()
    ==
    0
)


actual_test_counts = (
    test_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_test_counts
    ==
    EXPECTED_TEST_COUNTS
), (
    f"Unexpected test distribution.\n"
    f"Expected: {EXPECTED_TEST_COUNTS}\n"
    f"Actual:   {actual_test_counts}"
)


print("\n" + "=" * 80)
print("FINAL TEST-SET INTEGRITY CHECK")
print("=" * 80)


print(
    "Test samples:",
    len(test_df)
)


print(
    "Missing tweets:",
    test_df[
        "tweet"
    ]
    .isna()
    .sum()
)


print(
    "Missing labels:",
    test_df[
        "label"
    ]
    .isna()
    .sum()
)


print(
    "\nTest class distribution:"
)


display(
    test_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nTest-set integrity check: PASSED"
)


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME,
        use_fast=True,
    )
)


# ============================================================
# TEST DATASET
#
# Reuse RUHSOLDDataset if it already exists.
# Otherwise define it here.
# ============================================================

if "RUHSOLDDataset" not in globals():

    from torch.utils.data import Dataset


    class RUHSOLDDataset(Dataset):

        def __init__(
            self,
            dataframe,
            tokenizer,
            max_length,
        ):

            self.texts = (
                dataframe[
                    "tweet"
                ]
                .astype(str)
                .tolist()
            )

            self.labels = (
                dataframe[
                    "label"
                ]
                .astype(int)
                .tolist()
            )

            self.tokenizer = tokenizer

            self.max_length = max_length


        def __len__(self):

            return len(
                self.labels
            )


        def __getitem__(
            self,
            idx,
        ):

            encoding = (
                self.tokenizer(
                    self.texts[idx],

                    truncation=True,

                    max_length=(
                        self.max_length
                    ),

                    padding=False,
                )
            )

            encoding[
                "labels"
            ] = (
                self.labels[idx]
            )

            return encoding


# ============================================================
# BUILD TEST DATASET ONCE
# ============================================================

test_dataset = RUHSOLDDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


assert (
    len(test_dataset)
    ==
    EXPECTED_TEST_SIZE
)


# ============================================================
# SAME DYNAMIC PADDING FOR EVERY MODEL
# ============================================================

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True,
    )
)


test_loader = DataLoader(
    test_dataset,

    batch_size=(
        TEST_BATCH_SIZE
    ),

    shuffle=False,

    collate_fn=(
        data_collator
    ),
)


print(
    "\nTest DataLoader samples:",
    len(test_dataset)
)


print(
    "Test inference batch size:",
    TEST_BATCH_SIZE
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "\nEvaluation device:",
    device
)


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_test_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )


    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(

        y_true,
        y_pred,

        labels=METRIC_LABELS,

        average="macro",

        zero_division=0,
    )


    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = precision_recall_fscore_support(

        y_true,
        y_pred,

        labels=METRIC_LABELS,

        average="weighted",

        zero_division=0,
    )


    return {

        "test_accuracy":
            accuracy,

        "test_macro_precision":
            macro_precision,

        "test_macro_recall":
            macro_recall,

        "test_macro_f1":
            macro_f1,

        "test_weighted_precision":
            weighted_precision,

        "test_weighted_recall":
            weighted_recall,

        "test_weighted_f1":
            weighted_f1,
    }


# ============================================================
# RESULT STORAGE
# ============================================================

all_overall_results = []

all_per_class_results = []

reference_test_labels = None


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

for condition, seed_map in FINAL_CHECKPOINTS.items():

    print("\n")
    print("#" * 90)
    print(
        f"FINAL TEST CONDITION: {condition}"
    )
    print("#" * 90)


    condition_results_dir = (
        FINAL_TEST_RESULTS_ROOT
        /
        condition
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )


    condition_results_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    for seed, checkpoint_path in seed_map.items():

        print("\n" + "=" * 80)

        print(
            f"{condition.upper()} "
            f"- FINAL TEST - SEED {seed}"
        )

        print("=" * 80)


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        # ====================================================
        # LOAD TRAINED CHECKPOINT NORMALLY
        #
        # IMPORTANT:
        # No dtype conversion is specified here.
        # Evaluation uses the same FP32 loading for all
        # nine XLM-R checkpoints.
        # ====================================================

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                checkpoint_path
            )
        )


        model = model.to(
            device
        )


        model.eval()


        model_dtype = (
            next(
                model.parameters()
            ).dtype
        )


        print(
            "Loaded checkpoint:"
        )

        print(
            checkpoint_path
        )


        print(
            "Model device:",
            next(
                model.parameters()
            ).device
        )


        print(
            "Model dtype:",
            model_dtype
        )


        assert (
            model_dtype
            ==
            torch.float32
        ), (
            f"Expected FP32 test loading, "
            f"found {model_dtype}"
        )


        # ====================================================
        # INFERENCE
        # ====================================================

        seed_true = []

        seed_pred = []


        with torch.inference_mode():

            for batch in test_loader:

                labels = (
                    batch[
                        "labels"
                    ]
                    .cpu()
                    .numpy()
                )


                model_inputs = {

                    key:
                        value.to(
                            device
                        )

                    for key, value
                    in batch.items()

                    if key != "labels"
                }


                outputs = model(
                    **model_inputs
                )


                predictions = (
                    torch.argmax(
                        outputs.logits,
                        dim=-1,
                    )
                    .cpu()
                    .numpy()
                )


                seed_true.extend(
                    labels.tolist()
                )


                seed_pred.extend(
                    predictions.tolist()
                )


        seed_true = np.asarray(
            seed_true,
            dtype=int,
        )


        seed_pred = np.asarray(
            seed_pred,
            dtype=int,
        )


        # ====================================================
        # TEST INTEGRITY
        # ====================================================

        assert (
            len(seed_true)
            ==
            EXPECTED_TEST_SIZE
        )


        assert (
            len(seed_pred)
            ==
            EXPECTED_TEST_SIZE
        )


        assert np.array_equal(
            seed_true,
            test_df[
                "label"
            ]
            .to_numpy(
                dtype=int
            ),
        )


        if reference_test_labels is None:

            reference_test_labels = (
                seed_true.copy()
            )

        else:

            assert np.array_equal(
                reference_test_labels,
                seed_true,
            )


        print(
            f"\nSeed {seed} test integrity check: PASSED"
        )


        print(
            "Test predictions completed:",
            len(seed_pred)
        )


        # ====================================================
        # OVERALL TEST METRICS
        # ====================================================

        metrics = (
            calculate_test_metrics(
                seed_true,
                seed_pred,
            )
        )


        print("\n" + "-" * 60)

        print(
            f"FINAL TEST RESULTS "
            f"- {condition} - SEED {seed}"
        )

        print("-" * 60)


        print(
            "Accuracy        :",
            f'{metrics["test_accuracy"]:.4f}'
        )

        print(
            "Macro Precision :",
            f'{metrics["test_macro_precision"]:.4f}'
        )

        print(
            "Macro Recall    :",
            f'{metrics["test_macro_recall"]:.4f}'
        )

        print(
            "Macro F1        :",
            f'{metrics["test_macro_f1"]:.4f}'
        )

        print(
            "Weighted F1     :",
            f'{metrics["test_weighted_f1"]:.4f}'
        )


        # ====================================================
        # CLASSIFICATION REPORT
        # ====================================================

        report = classification_report(

            seed_true,
            seed_pred,

            labels=METRIC_LABELS,

            target_names=[
                id2label[
                    class_id
                ]
                for class_id
                in METRIC_LABELS
            ],

            output_dict=True,

            zero_division=0,
        )


        report_df = (
            pd.DataFrame(
                report
            )
            .transpose()
        )


        print(
            f"\nPer-class test results "
            f"- {condition} - seed {seed}:"
        )


        display(
            report_df.round(4)
        )


        # ====================================================
        # STORE OVERALL RESULT
        # ====================================================

        overall_row = {

            "condition":
                condition,

            "seed":
                seed,

            "checkpoint":
                str(
                    checkpoint_path
                ),

            "test_samples":
                EXPECTED_TEST_SIZE,

            **metrics,
        }


        all_overall_results.append(
            overall_row
        )


        # ====================================================
        # STORE PER-CLASS RESULT
        # ====================================================

        for class_id in METRIC_LABELS:

            class_name = (
                id2label[
                    class_id
                ]
            )


            class_metrics = (
                report[
                    class_name
                ]
            )


            all_per_class_results.append({

                "condition":
                    condition,

                "seed":
                    seed,

                "class_id":
                    class_id,

                "class_name":
                    class_name,

                "precision":
                    class_metrics[
                        "precision"
                    ],

                "recall":
                    class_metrics[
                        "recall"
                    ],

                "f1":
                    class_metrics[
                        "f1-score"
                    ],

                "support":
                    class_metrics[
                        "support"
                    ],
            })


        # ====================================================
        # SAVE PREDICTIONS
        # ====================================================

        prediction_df = pd.DataFrame({

            "tweet":
                test_df[
                    "tweet"
                ],

            "true_label_id":
                seed_true,

            "predicted_label_id":
                seed_pred,

            "true_label":
                [
                    id2label[
                        int(label)
                    ]
                    for label
                    in seed_true
                ],

            "predicted_label":
                [
                    id2label[
                        int(label)
                    ]
                    for label
                    in seed_pred
                ],
        })


        prediction_df.to_csv(

            condition_results_dir
            /
            f"seed_{seed}_test_predictions.csv",

            index=False,
        )


        report_df.to_csv(

            condition_results_dir
            /
            f"seed_{seed}_classification_report.csv"
        )


        # ====================================================
        # MEMORY CLEANUP
        # ====================================================

        del outputs
        del model


        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ============================================================
# CROSS-MODEL TEST-LABEL CONSISTENCY
# ============================================================

print("\n" + "=" * 80)

print(
    "CROSS-MODEL TEST-LABEL CONSISTENCY: PASSED"
)

print("=" * 80)


# ============================================================
# OVERALL RESULTS DATAFRAME
# ============================================================

overall_results_df = (
    pd.DataFrame(
        all_overall_results
    )
)


per_class_results_df = (
    pd.DataFrame(
        all_per_class_results
    )
)


assert (
    len(
        overall_results_df
    )
    ==
    9
)


assert (
    len(
        per_class_results_df
    )
    ==
    45
)


# ============================================================
# THREE-SEED CONDITION SUMMARY
# ============================================================

condition_summary_df = (

    overall_results_df

    .groupby(
        "condition"
    )

    .agg(

        accuracy_mean=(
            "test_accuracy",
            "mean"
        ),

        accuracy_std=(
            "test_accuracy",
            "std"
        ),

        macro_precision_mean=(
            "test_macro_precision",
            "mean"
        ),

        macro_precision_std=(
            "test_macro_precision",
            "std"
        ),

        macro_recall_mean=(
            "test_macro_recall",
            "mean"
        ),

        macro_recall_std=(
            "test_macro_recall",
            "std"
        ),

        macro_f1_mean=(
            "test_macro_f1",
            "mean"
        ),

        macro_f1_std=(
            "test_macro_f1",
            "std"
        ),

        weighted_f1_mean=(
            "test_weighted_f1",
            "mean"
        ),

        weighted_f1_std=(
            "test_weighted_f1",
            "std"
        ),
    )

    .reset_index()
)


# ============================================================
# THREE-SEED PER-CLASS SUMMARY
# ============================================================

per_class_summary_df = (

    per_class_results_df

    .groupby(
        [
            "condition",
            "class_id",
            "class_name",
        ]
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )

    .reset_index()
)


# ============================================================
# DISPLAY FINAL COMPARISON
# ============================================================

print("\n")
print("=" * 90)

print(
    "FINAL CORRECTED XLM-R TEST COMPARISON"
)

print("=" * 90)


print(
    "\nAll nine test runs:"
)


display(
    overall_results_df.round(4)
)


print(
    "\nThree-seed condition summary:"
)


display(
    condition_summary_df.round(4)
)


print(
    "\nThree-seed per-class test summary:"
)


display(
    per_class_summary_df.round(4)
)


# ============================================================
# MACRO-F1 SUMMARY
# ============================================================

print("\n" + "=" * 90)

print(
    "FINAL TEST MACRO-F1 SUMMARY"
)

print("=" * 90)


for _, row in (
    condition_summary_df
    .iterrows()
):

    print(
        f'{row["condition"]}: '
        f'{row["macro_f1_mean"]:.4f} '
        f'± {row["macro_f1_std"]:.4f}'
    )


# ============================================================
# SAVE FINAL RESULTS
# ============================================================

overall_results_df.to_csv(

    FINAL_TEST_RESULTS_ROOT
    / "all_xlmr_test_runs.csv",

    index=False,
)


condition_summary_df.to_csv(

    FINAL_TEST_RESULTS_ROOT
    / "xlmr_test_condition_summary.csv",

    index=False,
)


per_class_results_df.to_csv(

    FINAL_TEST_RESULTS_ROOT
    / "all_xlmr_per_class_test_results.csv",

    index=False,
)


per_class_summary_df.to_csv(

    FINAL_TEST_RESULTS_ROOT
    / "xlmr_per_class_test_summary.csv",

    index=False,
)


# ============================================================
# FINAL COMPLETION
# ============================================================

print("\n" + "=" * 90)

print(
    "FINAL FAIR XLM-R TEST EVALUATION COMPLETE"
)

print("=" * 90)


print(
    "Conditions evaluated:",
    list(
        FINAL_CHECKPOINTS.keys()
    )
)


print(
    "Seeds per condition:",
    [
        42,
        43,
        44,
    ]
)


print(
    "Test samples per run:",
    EXPECTED_TEST_SIZE
)


print(
    "\nAll final comparison results saved to:"
)

print(
    FINAL_TEST_RESULTS_ROOT
)

[HAMI-core Msg(155:139666446445888:libvgpu.c:839)]: Initializing.....


FINAL XLM-R CHECKPOINT AUDIT

Original XLM-R
Seed 42: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42/checkpoint-2005
Seed 43: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/seed_43/checkpoint-2005
Seed 44: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/seed_44/checkpoint-1203

Unfiltered 1x
Seed 42: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42/checkpoint-4419
Seed 43: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_43/checkpoint-1964
Seed 44: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_44/checkpoint-1473

QC-Filtered 1x
Seed 42: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_filtered_1x_corrected/seed_42/checkpoint-982
Seed 43: True /home/

,label,count
0,0,481
1,1,1070
2,2,156
3,3,168
4,4,128



Test-set integrity check: PASSED

Test DataLoader samples: 2003
Test inference batch size: 4

Evaluation device: cuda


##########################################################################################
FINAL TEST CONDITION: Original XLM-R
##########################################################################################

ORIGINAL XLM-R - FINAL TEST - SEED 42


[HAMI-core Msg(155:139666446445888:libvgpu.c:855)]: Initialized


...


Three-seed condition summary:


,condition,accuracy_mean,accuracy_std,macro_precision_mean,macro_precision_std,macro_recall_mean,macro_recall_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,Original XLM-R,0.7943,0.0058,0.7156,0.0068,0.7080,0.0112,0.7078,0.0113,0.7931,0.0071
1,QC-Filtered 1x,0.7903,0.0182,0.6989,0.0252,0.7212,0.0311,0.7071,0.0292,0.7897,0.0218
2,Unfiltered 1x,0.7815,0.0118,0.6945,0.0327,0.7027,0.0145,0.6899,0.0096,0.7798,0.0074



Three-seed per-class test summary:


,condition,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support
0,Original XLM-R,0,Abusive/Offensive,0.7202,0.0181,0.6667,0.0526,0.6911,0.0212,481.0
1,Original XLM-R,1,Normal,0.8887,0.0130,0.9100,0.0186,0.8990,0.0030,1070.0
2,Original XLM-R,2,Religious Hate,0.7069,0.0694,0.6538,0.0611,0.6754,0.0169,156.0
3,Original XLM-R,3,Sexism,0.6311,0.0926,0.6766,0.0821,0.6453,0.0144,168.0
4,Original XLM-R,4,Profane,0.6311,0.0813,0.6328,0.0435,0.6282,0.0216,128.0
5,QC-Filtered 1x,0,Abusive/Offensive,0.7297,0.0182,0.6452,0.0782,0.6829,0.0440,481.0
6,QC-Filtered 1x,1,Normal,0.8917,0.0285,0.8994,0.0143,0.8952,0.0077,1070.0
7,QC-Filtered 1x,2,Religious Hate,0.6597,0.0265,0.7521,0.0074,0.7028,0.0182,156.0
8,QC-Filtered 1x,3,Sexism,0.6246,0.0298,0.6349,0.0756,0.6286,0.0483,168.0
9,QC-Filtered 1x,4,Profane,0.5886,0.0693,0.6745,0.0401,0.6260,0.0310,128.0



FINAL TEST MACRO-F1 SUMMARY
Original XLM-R: 0.7078 ± 0.0113
QC-Filtered 1x: 0.7071 ± 0.0292
Unfiltered 1x: 0.6899 ± 0.0096

FINAL FAIR XLM-R TEST EVALUATION COMPLETE
Conditions evaluated: ['Original XLM-R', 'Unfiltered 1x', 'QC-Filtered 1x']
Seeds per condition: [42, 43, 44]
Test samples per run: 2003

All final comparison results saved to:
/home/jovyan/project work/data_analyssis/classifier/outputs/final_corrected_xlmr_comparison
